<a href="https://colab.research.google.com/github/Pensive1881/DSR44_2025/blob/main/scratchpad_clip_plus_detection_and_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
!uv pip install fiftyone==1.9.0

## Obtain the CLIP model

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

clip_model = foz.load_zoo_model('clip-vit-base32-torch')


/usr/local/lib/python3.12/dist-packages/glob2/fnmatch.py:141: SyntaxWarning: invalid escape sequence '\Z'
  return '(?ms)' + res + '\Z'


INFO:fiftyone.core.models:Downloading model from 'https://openaipublic.azureedge.net/clip/models/40d365715913c9da98579312b702a82c18be219cc2a73407c4526f58eba950af/ViT-B-32.pt'...


 100% |██████|    2.6Gb/2.6Gb [748.1ms elapsed, 0s remaining, 3.5Gb/s]      


INFO:eta.core.utils: 100% |██████|    2.6Gb/2.6Gb [748.1ms elapsed, 0s remaining, 3.5Gb/s]      


/usr/local/lib/python3.12/dist-packages/fiftyone/utils/clip/tokenizer.py:107: SyntaxWarning: invalid escape sequence '\p'
  + """[\p{L}]+|[\p{N}]|[^\s\p{L}\p{N}]+""",
INFO:fiftyone.utils.clip.zoo:Downloading CLIP tokenizer...


 100% |█████|   10.4Mb/10.4Mb [8.9ms elapsed, 0s remaining, 1.1Gb/s]       


INFO:eta.core.utils: 100% |█████|   10.4Mb/10.4Mb [8.9ms elapsed, 0s remaining, 1.1Gb/s]       


In [ ]:
from google.colab import drive
drive.mount('/gdrive')
%cd /gdrive

Mounted at /gdrive
/gdrive


## Create the FiftyOne dataset with our custom artwork

In [ ]:
from pathlib import Path
import os
artist_name = 'Hokusai'
path = Path(f'/gdrive/MyDrive/art_recommendation/{artist_name}')

In [ ]:
# here 'paintings' should appear
os.listdir(path)

['paintings', 'Hokusai artworks', 'paintings_embeddings.pickle', 'images.zip']

In [ ]:
len(os.listdir(path / 'paintings'))

832

In [ ]:
import fiftyone as fo

dataset_name = f"{artist_name}_paintings"

# delete the dataset in case it exists already on the Colab instance
# (due to multiple evaluations of the code cell)
if fo.dataset_exists(dataset_name):
  fo.delete_dataset(dataset_name)

You are running the oldest supported major version of MongoDB. Please refer to https://deprecation.voxel51.com for deprecation notices. You can suppress this exception by setting your `database_validation` config parameter to `False`. See https://docs.voxel51.com/user_guide/config.html#configuring-a-mongodb-connection for more information


In [ ]:
# this creates an empty dataset
dataset = fo.Dataset(dataset_name)
dataset

Name:        Hokusai_paintings
Media type:  None
Num samples: 0
Persistent:  False
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.Metadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField

In [ ]:
images_dir = path / 'paintings'

In [ ]:
# We use the location of the images_dir to add samples to the dataset
dataset.add_dir(images_dir, dataset_type=fo.types.ImageDirectory)

 100% |█████████████████| 832/832 [152.2ms elapsed, 0s remaining, 5.5K samples/s]  


INFO:eta.core.utils: 100% |█████████████████| 832/832 [152.2ms elapsed, 0s remaining, 5.5K samples/s]  


['6903817d9360655994a6dad3',
 '6903817d9360655994a6dad4',
 '6903817d9360655994a6dad5',
 '6903817d9360655994a6dad6',
 '6903817d9360655994a6dad7',
 '6903817d9360655994a6dad8',
 '6903817d9360655994a6dad9',
 '6903817d9360655994a6dada',
 '6903817d9360655994a6dadb',
 '6903817d9360655994a6dadc',
 '6903817d9360655994a6dadd',
 '6903817d9360655994a6dade',
 '6903817d9360655994a6dadf',
 '6903817d9360655994a6dae0',
 '6903817d9360655994a6dae1',
 '6903817d9360655994a6dae2',
 '6903817d9360655994a6dae3',
 '6903817d9360655994a6dae4',
 '6903817d9360655994a6dae5',
 '6903817d9360655994a6dae6',
 '6903817d9360655994a6dae7',
 '6903817d9360655994a6dae8',
 '6903817d9360655994a6dae9',
 '6903817d9360655994a6daea',
 '6903817d9360655994a6daeb',
 '6903817d9360655994a6daec',
 '6903817d9360655994a6daed',
 '6903817d9360655994a6daee',
 '6903817d9360655994a6daef',
 '6903817d9360655994a6daf0',
 '6903817d9360655994a6daf1',
 '6903817d9360655994a6daf2',
 '6903817d9360655994a6daf3',
 '6903817d9360655994a6daf4',
 '6903817d9360

In [ ]:
# add metadata on file size, image format, and image dimensions
#dataset.compute_metadata()

Computing metadata...


INFO:fiftyone.core.metadata:Computing metadata...


 100% |█████████████████| 832/832 [30.2s elapsed, 0s remaining, 504.4 samples/s]     


INFO:eta.core.utils: 100% |█████████████████| 832/832 [30.2s elapsed, 0s remaining, 504.4 samples/s]     


## Compute CLIP embeddings



In [ ]:
import fiftyone.brain as fob

image_index = fob.compute_similarity(
    dataset,
    model="clip-vit-base32-torch",
    brain_key="clip_img_sim",
    embeddings='clip_embeddings'
)

Computing embeddings...


INFO:fiftyone.brain.internal.core.utils:Computing embeddings...


 100% |█████████████████| 832/832 [25.0s elapsed, 0s remaining, 10.0 samples/s]      


INFO:eta.core.utils: 100% |█████████████████| 832/832 [25.0s elapsed, 0s remaining, 10.0 samples/s]      


In [ ]:
session = fo.launch_app(dataset, auto=False)
print(session.url)

Session launched. Run `session.show()` to open the App in a cell output.


INFO:fiftyone.core.session.session:Session launched. Run `session.show()` to open the App in a cell output.



Welcome to

███████╗██╗███████╗████████╗██╗   ██╗ ██████╗ ███╗   ██╗███████╗
██╔════╝██║██╔════╝╚══██╔══╝╚██╗ ██╔╝██╔═══██╗████╗  ██║██╔════╝
█████╗  ██║█████╗     ██║    ╚████╔╝ ██║   ██║██╔██╗ ██║█████╗
██╔══╝  ██║██╔══╝     ██║     ╚██╔╝  ██║   ██║██║╚██╗██║██╔══╝
██║     ██║██║        ██║      ██║   ╚██████╔╝██║ ╚████║███████╗
╚═╝     ╚═╝╚═╝        ╚═╝      ╚═╝    ╚═════╝ ╚═╝  ╚═══╝╚══════╝ v1.9.0

If you're finding FiftyOne helpful, here's how you can get involved:

|
|  ⭐⭐⭐ Give the project a star on GitHub ⭐⭐⭐
|  https://github.com/voxel51/fiftyone
|
|  🚀🚀🚀 Join the FiftyOne Discord community 🚀🚀🚀
|  https://community.voxel51.com/
|



INFO:fiftyone.core.session.session:
Welcome to

███████╗██╗███████╗████████╗██╗   ██╗ ██████╗ ███╗   ██╗███████╗
██╔════╝██║██╔════╝╚══██╔══╝╚██╗ ██╔╝██╔═══██╗████╗  ██║██╔════╝
█████╗  ██║█████╗     ██║    ╚████╔╝ ██║   ██║██╔██╗ ██║█████╗
██╔══╝  ██║██╔══╝     ██║     ╚██╔╝  ██║   ██║██║╚██╗██║██╔══╝
██║     ██║██║        ██║      ██║   ╚██████╔╝██║ ╚████║███████╗
╚═╝     ╚═╝╚═╝        ╚═╝      ╚═╝    ╚═════╝ ╚═╝  ╚═══╝╚══════╝ v1.9.0

If you're finding FiftyOne helpful, here's how you can get involved:

|
|  ⭐⭐⭐ Give the project a star on GitHub ⭐⭐⭐
|  https://github.com/voxel51/fiftyone
|
|  🚀🚀🚀 Join the FiftyOne Discord community 🚀🚀🚀
|  https://community.voxel51.com/
|



https://5151-gpu-a100-hm-3htvxwhnfiruh-c.asia-southeast1-1.prod.colab.dev?polling=true


In [ ]:
image_index.config.support_prompts
